# SCBE-AETHERMOORE Cloud Workspace

**Purpose**: Remote development workspace for SCBE-AETHERMOORE when your local PC is resource-constrained.

This notebook provides:
1. Repo setup and dependency installation
2. Semantic mesh self-test validation
3. Training data pipeline inspection
4. HuggingFace Hub upload for training data
5. Fine-tuning with free Colab GPU (T4)
6. SCBE API server with ngrok tunnel
7. n8n workflow automation from Colab

**GPU**: Enable GPU via `Runtime > Change runtime type > T4 GPU` for fine-tuning cells.

---

### Architecture Reference

| Layers | Function |
|--------|----------|
| L1-4 | Complex Context -> Realification -> Weighted Transform -> Poincare Embedding |
| L5 | Hyperbolic distance (invariant metric) |
| L6-7 | Breathing Transform + Mobius Phase |
| L8 | Multi-Well Realms |
| L9-10 | Spectral + Spin Coherence |
| L11 | Triadic Temporal Distance |
| L12 | Harmonic Wall H(d,R) = R^(d^2) |
| L13 | Decision Gate: ALLOW / QUARANTINE / DENY |
| L14 | Audio Axis (FFT telemetry) |

---
## 1. Setup -- Clone Repo and Install Dependencies

Clones the SCBE-AETHERMOORE repository and installs all required Python packages.
This cell only needs to run once per Colab session.

In [ ]:
import os

# Only clone if not already present
if not os.path.exists('SCBE-AETHERMOORE'):
    !git clone https://github.com/issdandavis/SCBE-AETHERMOORE.git
else:
    print('Repo already cloned -- pulling latest changes...')
    !cd SCBE-AETHERMOORE && git pull

%cd SCBE-AETHERMOORE

# Install core dependencies
!pip install -q fastapi uvicorn pydantic numpy scipy

# Install full project requirements (skip liboqs if build fails on Colab)
!pip install -q -r requirements.txt 2>/dev/null || echo 'Some optional deps failed -- core deps installed above'

# Verify installation
import numpy as np
import scipy
print(f'numpy:  {np.__version__}')
print(f'scipy:  {scipy.__version__}')
print('Setup complete.')

---
## 2. Semantic Mesh Self-Test

Runs the MCP server self-test to validate that the semantic mesh, axiom pipeline,
and core SCBE components are functioning correctly in this environment.

In [ ]:
# Run the semantic mesh self-test
!python -m src.mcp_server.selftest

# If the selftest module isn't available, run the axiom-level validation instead
import subprocess
result = subprocess.run(
    ['python', '-m', 'src.mcp_server.selftest'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('selftest module not found -- running axiom validation fallback...')
    print()

    # Fallback: run the axiom grouped benchmarks
    axiom_paths = [
        'symphonic_cipher/scbe_aethermoore/axiom_grouped/langues_metric.py',
        'symphonic_cipher/scbe_aethermoore/axiom_grouped/benchmark_comparison.py',
    ]
    for p in axiom_paths:
        if os.path.exists(p):
            print(f'--- Running {p} ---')
            !python {p}
            print()

    # Also try pytest quick smoke tests
    print('--- Running quick smoke tests ---')
    !python -m pytest tests/ -m homebrew --tb=short -q 2>/dev/null || echo 'No homebrew tests found or pytest not configured'

---
## 3. Training Data Pipeline

Loads and validates all JSONL training data files. Displays pair counts per file
and validates JSON structure. The training data covers architecture, tongues, math,
game design, lore, music, and gacha sessions.

In [ ]:
import json
import glob
import os

files = sorted(glob.glob('training-data/*.jsonl'))

if not files:
    print('No training data files found in training-data/*.jsonl')
    print('Checking alternative locations...')
    for alt in ['data/*.jsonl', '**/*.jsonl']:
        alt_files = glob.glob(alt, recursive=True)
        if alt_files:
            print(f'Found {len(alt_files)} JSONL files via pattern: {alt}')
            files = sorted(alt_files)
            break

total = 0
errors = 0
categories = {}

print(f'{"Pairs":>6}  {"Errors":>6}  File')
print('-' * 60)

for f in files:
    file_pairs = 0
    file_errors = 0
    with open(f) as fh:
        for line_num, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                file_pairs += 1
            except json.JSONDecodeError as e:
                file_errors += 1
                if file_errors <= 3:  # Only show first 3 errors per file
                    print(f'  JSON error in {f}:{line_num}: {e}')
    total += file_pairs
    errors += file_errors

    # Categorize by filename pattern
    basename = os.path.basename(f).replace('.jsonl', '')
    category = basename.rsplit('_', 1)[0] if '_' in basename else basename
    categories[category] = categories.get(category, 0) + file_pairs

    print(f'{file_pairs:6d}  {file_errors:6d}  {f}')

print('-' * 60)
print(f'{total:6d}  {errors:6d}  TOTAL ({len(files)} files)')
print()

if categories:
    print('Category breakdown:')
    for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
        print(f'  {count:4d}  {cat}')

In [ ]:
# Inspect a sample training pair from the first file
if files:
    sample_file = files[0]
    with open(sample_file) as fh:
        first_line = fh.readline().strip()
        if first_line:
            sample = json.loads(first_line)
            print(f'Sample from: {sample_file}')
            print(f'Keys: {list(sample.keys())}')
            print()
            print(json.dumps(sample, indent=2)[:2000])  # Truncate long samples
else:
    print('No training data files to inspect.')

---
## 4. HuggingFace Hub Upload

Pushes training data to HuggingFace Hub. Set your token in Colab Secrets:
1. Click the key icon in the left sidebar
2. Add a secret named `HF_TOKEN` with your HuggingFace write token
3. Toggle notebook access to ON

Get a token at: https://huggingface.co/settings/tokens

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import HfApi
import os
import glob

# Attempt to load token from Colab secrets first, then env var
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('Loaded HF_TOKEN from Colab secrets.')
except (ImportError, Exception):
    hf_token = os.environ.get('HF_TOKEN', '')
    if not hf_token:
        print('WARNING: No HF_TOKEN found.')
        print('Set it in Colab secrets (key icon in sidebar) or run:')
        print('  os.environ["HF_TOKEN"] = "hf_your_token_here"')
        print('Then re-run this cell.')

REPO_ID = 'issdandavis/aethermoor-npc-v1'

if hf_token:
    api = HfApi(token=hf_token)

    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True)
        print(f'Repository ready: https://huggingface.co/{REPO_ID}')
    except Exception as e:
        print(f'Repo creation note: {e}')

    # Upload all training data files
    training_files = glob.glob('training-data/*.jsonl')
    uploaded = 0
    for f in training_files:
        try:
            api.upload_file(
                path_or_fileobj=f,
                path_in_repo=f'training-data/{os.path.basename(f)}',
                repo_id=REPO_ID,
                repo_type='model',
            )
            uploaded += 1
            print(f'  Uploaded: {os.path.basename(f)}')
        except Exception as e:
            print(f'  Failed:   {os.path.basename(f)} -- {e}')

    print(f'\nUploaded {uploaded}/{len(training_files)} files to {REPO_ID}')
else:
    print('Skipping upload -- no token configured.')

---
## 5. Fine-Tuning (Free Colab GPU)

Supervised fine-tuning of a small language model on SCBE training data.
Uses QLoRA (4-bit quantization) to fit within Colab's free T4 GPU (16GB VRAM).

**Recommended base models for free tier:**
- `microsoft/phi-2` (2.7B) -- good quality, fits T4
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0` -- smallest, fastest
- `Qwen/Qwen2.5-1.5B-Instruct` -- strong for size

**Important**: Make sure GPU is enabled: `Runtime > Change runtime type > T4 GPU`

In [ ]:
# Install fine-tuning dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q torch  # Usually pre-installed on Colab

import torch
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Enable via Runtime > Change runtime type > T4 GPU')

In [ ]:
import json
import glob
from datasets import Dataset

# Load all training data into a unified dataset
all_pairs = []
for f in sorted(glob.glob('training-data/*.jsonl')):
    with open(f) as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    pair = json.loads(line)
                    all_pairs.append(pair)
                except json.JSONDecodeError:
                    pass

print(f'Loaded {len(all_pairs)} training pairs')

if all_pairs:
    # Detect format and normalize to instruction/response
    sample_keys = set(all_pairs[0].keys())
    print(f'Format keys: {sample_keys}')

    def format_pair(pair):
        """Normalize various JSONL formats to a single text field for SFT."""
        # ChatML / messages format
        if 'messages' in pair:
            msgs = pair['messages']
            text = ''
            for m in msgs:
                role = m.get('role', 'user')
                content = m.get('content', '')
                text += f'<|{role}|>\n{content}\n'
            return {'text': text.strip()}

        # Instruction / input / output format
        if 'instruction' in pair:
            inp = pair.get('input', '')
            out = pair.get('output', pair.get('response', ''))
            instr = pair['instruction']
            if inp:
                text = f'<|system|>\n{instr}\n<|user|>\n{inp}\n<|assistant|>\n{out}'
            else:
                text = f'<|user|>\n{instr}\n<|assistant|>\n{out}'
            return {'text': text}

        # Prompt / completion format
        if 'prompt' in pair:
            return {'text': f'<|user|>\n{pair["prompt"]}\n<|assistant|>\n{pair.get("completion", pair.get("response", ""))}'}

        # Fallback: concatenate all values
        return {'text': ' '.join(str(v) for v in pair.values())}

    formatted = [format_pair(p) for p in all_pairs]
    dataset = Dataset.from_list(formatted)
    print(f'Dataset size: {len(dataset)}')
    print(f'Sample (truncated): {dataset[0]["text"][:300]}...')
else:
    print('No training data found. Upload data to training-data/ first.')
    dataset = None

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ---- Configuration ----
BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # Change as needed
OUTPUT_DIR = './scbe-finetuned'
MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4

# ---- Quantization config (4-bit for T4) ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ---- Load model and tokenizer ----
print(f'Loading {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# ---- LoRA config ----
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---- Training arguments ----
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    report_to='none',
)

# ---- Train ----
if dataset is not None and len(dataset) > 0:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=training_args,
        max_seq_length=MAX_SEQ_LENGTH,
    )

    print(f'Starting training: {len(dataset)} samples, {NUM_EPOCHS} epochs')
    trainer.train()

    # Save the LoRA adapter
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f'Model saved to {OUTPUT_DIR}')
else:
    print('No dataset available. Load training data first (Section 3).')

In [ ]:
# Quick inference test with the fine-tuned model
if os.path.exists(OUTPUT_DIR):
    from peft import PeftModel

    # Reload base + adapter for inference
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    finetuned = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    finetuned.eval()

    test_prompts = [
        'What is the Harmonic Wall in SCBE-AETHERMOORE?',
        'Explain the Six Sacred Tongues.',
        'How does the Poincare ball model provide AI safety?',
    ]

    for prompt in test_prompts:
        inputs = tokenizer(f'<|user|>\n{prompt}\n<|assistant|>\n', return_tensors='pt').to('cuda')
        with torch.no_grad():
            outputs = finetuned.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
            )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f'Q: {prompt}')
        print(f'A: {response}')
        print('-' * 60)
else:
    print('No fine-tuned model found. Run the training cell first.')

In [ ]:
# Upload fine-tuned adapter to HuggingFace Hub
if os.path.exists(OUTPUT_DIR) and hf_token:
    from huggingface_hub import HfApi

    api = HfApi(token=hf_token)
    api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True)

    api.upload_folder(
        folder_path=OUTPUT_DIR,
        path_in_repo='adapter',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print(f'Adapter uploaded to https://huggingface.co/{REPO_ID}/tree/main/adapter')
else:
    print('Either no model to upload or no HF token configured.')

---
## 5b. HYDRA Multi-Model Training (6 Sacred Tongue Heads)

Train 6 specialized models — one per Sacred Tongue HYDRA head — each fine-tuned
for its specific role in the browser swarm:

| Tongue | Role | Base Model | Size |
|--------|------|-----------|------|
| **KO** | Scout/Commander | TinyLlama 1.1B | 1.1 GB |
| **AV** | Vision/Navigator | Moondream2 | 1.6 GB |
| **RU** | Reader/Policy | Phi-2 | 2.7 GB |
| **CA** | Clicker/Compute | Qwen2.5-1.5B | 1.5 GB |
| **UM** | Typer/Shadow | SmolLM-1.7B | 1.7 GB |
| **DR** | Judge/Schema | TinyLlama 1.1B | 1.1 GB |

All models use QLoRA 4-bit quantization to fit on free Colab T4 (16 GB VRAM).
Training data is filtered per-head using the Sacred Tongue category mappings.

Uses **Federated Multi-Tiered 6D Analysis** governance:
- Tier 1 (Local): TriManifoldLattice viability check per training pair
- Tier 2 (Regional): Matrix pulse across each head's data batch
- Tier 3 (Global): Byzantine 4/6 consensus across all heads

In [ ]:
# HYDRA Multi-Model Training — uses training/vertex_hydra_trainer.py
# Pick which head to train (change TONGUE to: KO, AV, RU, CA, UM, DR)

TONGUE = 'KO'  # @param ["KO", "RU", "CA", "UM", "DR"] {type:"string"}
PUSH_TO_HF = False  # @param {type:"boolean"}

# Run the trainer
import subprocess, os

cmd = ['python', 'training/vertex_hydra_trainer.py', '--head', TONGUE]
if PUSH_TO_HF:
    cmd.append('--push')

print(f'Training HYDRA {TONGUE} head...')
print(f'Command: {" ".join(cmd)}')
print('=' * 60)

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print(f'\nHYDRA {TONGUE} head training complete!')
    output_dir = f'./hydra-models/hydra-{TONGUE.lower()}-*'
    import glob
    dirs = glob.glob(output_dir)
    if dirs:
        print(f'Model saved to: {dirs[0]}')
else:
    print(f'\nTraining failed with exit code {result.returncode}')

In [ ]:
# Dry-run: validate config, check data, show what would be trained
# No GPU needed — safe to run anywhere

!python training/vertex_hydra_trainer.py --dry-run

---
## 6. API Server (ngrok Tunnel)

Starts the SCBE FastAPI server from Colab and exposes it via ngrok for public access.

**ngrok Setup:**
1. Sign up at https://ngrok.com (free tier works)
2. Get your auth token from https://dashboard.ngrok.com/get-started/your-authtoken
3. Add it as a Colab secret named `NGROK_TOKEN`

In [ ]:
# Train ALL 6 HYDRA heads sequentially (takes ~2-3 hours on free T4)
# Each head loads a different base model, trains on tongue-specific data,
# and saves a separate LoRA adapter.

PUSH_ALL = False  # @param {type:"boolean"}

import subprocess

cmd = ['python', 'training/vertex_hydra_trainer.py', '--all']
if PUSH_ALL:
    cmd.append('--push')

print('Training all 6 HYDRA Sacred Tongue heads...')
print('Expected order: KO -> AV (skip VLM) -> RU -> CA -> UM -> DR')
print('Each head: ~15-30 min on T4')
print('=' * 60)

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print('\nAll HYDRA heads trained!')
    import glob
    for d in sorted(glob.glob('./hydra-models/hydra-*')):
        print(f'  {d}')
else:
    print(f'\nTraining pipeline exited with code {result.returncode}')

In [ ]:
!pip install -q pyngrok

import os
import subprocess
import time
from pyngrok import ngrok, conf

# Load ngrok auth token
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_TOKEN')
except (ImportError, Exception):
    ngrok_token = os.environ.get('NGROK_TOKEN', '')

if not ngrok_token:
    print('WARNING: No NGROK_TOKEN found.')
    print('Set it in Colab secrets or run:')
    print('  os.environ["NGROK_TOKEN"] = "your_token_here"')
    print('Then re-run this cell.')
else:
    # Configure ngrok
    ngrok.set_auth_token(ngrok_token)

    API_PORT = 8000

    # Start the FastAPI server in the background
    api_process = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'src.api.main:app',
         '--host', '0.0.0.0', '--port', str(API_PORT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print(f'API server starting on port {API_PORT} (PID: {api_process.pid})')

    # Wait for server to start
    time.sleep(5)

    # Check if server is running
    if api_process.poll() is None:
        # Open ngrok tunnel
        public_url = ngrok.connect(API_PORT, 'http')
        print(f'\n--- SCBE API is live ---')
        print(f'Public URL:  {public_url}')
        print(f'Docs:        {public_url}/docs')
        print(f'Health:      {public_url}/health')
        print(f'\nThis URL is accessible from anywhere.')
        print(f'Use Ctrl+C or Runtime > Interrupt to stop.')
    else:
        print('API server failed to start. Checking error output...')
        stderr = api_process.stderr.read().decode()
        print(stderr[:2000])

In [ ]:
# Test the API endpoint
import requests

try:
    # Test local endpoint
    r = requests.get(f'http://localhost:{API_PORT}/health', timeout=5)
    print(f'Health check: {r.status_code}')
    print(r.json())
except Exception as e:
    print(f'Local health check failed: {e}')
    print('The API server may not be running. Check the cell above.')

In [ ]:
# Stop the API server and close ngrok tunnels
try:
    ngrok.kill()
    print('ngrok tunnels closed.')
except:
    pass

try:
    api_process.terminate()
    api_process.wait(timeout=5)
    print('API server stopped.')
except:
    print('API server was not running.')

---
## 7. n8n Workflow Automation

Installs and runs [n8n](https://n8n.io) directly from Colab with an ngrok tunnel
for webhook access. This lets you build automation workflows that connect to the
SCBE API, HuggingFace, GitHub, and other services.

**Note**: n8n requires Node.js 18+. Colab typically has Node.js pre-installed.

In [ ]:
# Check Node.js version and install n8n
!node --version
!npm install -g n8n 2>&1 | tail -5
print('n8n installed.')

In [ ]:
import subprocess
import time
import os
from pyngrok import ngrok

N8N_PORT = 5678

# Load ngrok token (reuse from earlier or load fresh)
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_TOKEN')
except (ImportError, Exception):
    ngrok_token = os.environ.get('NGROK_TOKEN', '')

if not ngrok_token:
    print('WARNING: No NGROK_TOKEN. n8n will only be accessible locally.')
    print('Set NGROK_TOKEN in Colab secrets for public webhook access.')

# Set n8n environment
env = os.environ.copy()
env['N8N_PORT'] = str(N8N_PORT)
env['N8N_PROTOCOL'] = 'http'
env['N8N_HOST'] = '0.0.0.0'
env['GENERIC_TIMEZONE'] = 'America/New_York'

# Start n8n in the background
n8n_process = subprocess.Popen(
    ['n8n', 'start'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(f'n8n starting on port {N8N_PORT} (PID: {n8n_process.pid})')

# Wait for n8n to initialize
time.sleep(10)

if n8n_process.poll() is None:
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        n8n_url = ngrok.connect(N8N_PORT, 'http')
        print(f'\n--- n8n is live ---')
        print(f'Public URL:    {n8n_url}')
        print(f'Webhook base:  {n8n_url}/webhook/')
        print(f'\nUse this webhook base URL in your n8n workflows.')
    else:
        print(f'\nn8n running locally at http://localhost:{N8N_PORT}')
        print('(Not publicly accessible without ngrok)')
else:
    print('n8n failed to start. Error output:')
    stderr = n8n_process.stderr.read().decode()
    print(stderr[:2000])

In [ ]:
# Stop n8n and close its tunnel
try:
    ngrok.kill()
    print('ngrok tunnels closed.')
except:
    pass

try:
    n8n_process.terminate()
    n8n_process.wait(timeout=5)
    print('n8n stopped.')
except:
    print('n8n was not running.')

---
## Utilities

Helpful cells for common operations during cloud development.

In [ ]:
# GPU memory monitoring
import torch

if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM:     {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB')
    print(f'Allocated:      {torch.cuda.memory_allocated() / 1e9:.2f} GB')
    print(f'Cached:         {torch.cuda.memory_reserved() / 1e9:.2f} GB')
    print(f'Free:           {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1e9:.2f} GB')
else:
    print('No GPU available.')
    !free -h  # Show system RAM instead

In [ ]:
# Mount Google Drive and save artifacts
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_BACKUP = '/content/drive/MyDrive/SCBE-AETHERMOORE-backup'

# Save fine-tuned model to Drive
if os.path.exists(OUTPUT_DIR):
    dest = f'{DRIVE_BACKUP}/scbe-finetuned'
    os.makedirs(dest, exist_ok=True)
    shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True)
    print(f'Model saved to Google Drive: {dest}')
else:
    print('No fine-tuned model to save.')

In [ ]:
# Check Colab disk space
!df -h /content
print()
!du -sh SCBE-AETHERMOORE/ 2>/dev/null || echo 'Repo not cloned yet'
!du -sh scbe-finetuned/ 2>/dev/null || echo 'No fine-tuned model yet'

In [ ]:
# Cleanup: free GPU memory
import gc
import torch

# Delete model objects if they exist
for var_name in ['model', 'base_model', 'finetuned', 'trainer']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated')
else:
    print('Cleanup complete.')